In [ ]:
import os
import sys
import pandas as pd
import numpy as np
import dotenv

import warnings
warnings.filterwarnings("ignore", category=pd.errors.PerformanceWarning)

dotenv.load_dotenv(dotenv.find_dotenv())

ROOT_PATH = os.getenv("ROOT_PATH")
MY_DATA_PATH = os.getenv("MY_DATA_PATH")
RAW_DATA_PATH = os.getenv("RAW_DATA_PATH")

sys.path.append(os.path.join(ROOT_PATH, "scripts"))

OUTPUT_FILEPATH = os.path.join(MY_DATA_PATH, "raw_data/alfin_unit_year_list.csv")

import alfin

In [2]:
df = []

cols = ['ID', 'SurveyYr', 'Name', 'FIPS Code-State', 'Type Code']
for y in range(2007, 2012):
    mydf = alfin._get_unit_data(y)
    mydf = mydf[cols]
    mydf = mydf.rename(columns={
        'ID': 'alfin_id', 
        'SurveyYr': 'year',
        'Name': 'entity_name',
        'FIPS Code-State': 'state_fips', 
        'Type Code': 'type_code'
    })
    mydf['year'] = 2000 + mydf['year'].astype(int)
    df.append(mydf)

cols = ['ID', 'SURVEY_YEAR', 'UNIT_TYPE_CODE', 'NAME', 'STATE_FIPS', 'FUNCTION_CODE']
for y in range(2012, 2024):
    mydf = alfin._get_unit_data(y)
    mydf = mydf[cols]
    mydf = mydf.rename(columns={
        'ID': 'alfin_id',
        'SURVEY_YEAR': 'year',
        'UNIT_TYPE_CODE': 'type_code',
        'NAME': 'entity_name',
        'STATE_FIPS': 'state_fips',
        'FUNCTION_CODE': 'function_code'
    })
    mydf = mydf.loc[mydf['entity_name'].str.len() > 0] # remove if name is empty
    mydf['year'] = 2000 + mydf['year'].astype(int)
    df.append(mydf)

df = pd.concat(df, ignore_index=True)
df['function_code'] = df['function_code'].fillna('').astype(str)
assert(df.duplicated(subset=['alfin_id', 'year']).sum() == 0)


In [3]:
df.to_csv(OUTPUT_FILEPATH, index=False)

In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 674737 entries, 0 to 674736
Data columns (total 6 columns):
 #   Column         Non-Null Count   Dtype 
---  ------         --------------   ----- 
 0   alfin_id       674737 non-null  object
 1   year           674737 non-null  int64 
 2   entity_name    674737 non-null  str   
 3   state_fips     674737 non-null  object
 4   type_code      674737 non-null  object
 5   function_code  674737 non-null  str   
dtypes: int64(1), object(3), str(2)
memory usage: 47.3+ MB
